### 배치 정규화

배치 정규화(batch nomalization)

#### 배치 정규화 알고리즘

1. 학습을 빨리 진행할 수 있음 (학습 속도 개선)

2. 초깃값에 크게 의존하지 않음 (골치 아픈 초깃값 선택 장애여 안녕!)
3. 오버피팅을 억제한다 (드롭아웃 등의 필요성 감소)

- 딥러닝의 학습 시간이 길다는 걸 생각하면 첫 번째 이점은 아주 반가운 일이다

- 초깃값에 크게 신경 쓸 필요가 없고, 오버피팅 억제 효과가 있다는 점도 딥러닝 학습의 두통거리를 덜어줌

- 배치 정규화의 기본 아이디어는 각 층에서의 활성화값이 적당히 분포되도록 조정하는 것

- 데이터 분포를 정규화 하는 '배치 정규화(Batch Norm) 계층'을 신경망에 삽입함


- 배치 정규화는 그 이름과 같이 학습 시 미니배치를 단위로 정규화 함

- 데이터 분포가 평균이 0, 분산이 1이 되도록 정규화 함

- 배치 정규화 계층마다 이 정규화된 데이터에 고유한 확대(Scale)와 이동(shift) 변환을 수행함

- 배치 정규화 알고리즘이 신경망에서 순전파 때 적용된다

#### 배치 정규화의 효과

- 거의 모든 경우에서 배치 정규화를 사용할 때의 학습 진도가 빠름

- 실제로 배치 정규화를 이용하지 않는 경우엔 초깃값이 잘 분포되어 있지 않으면 학습이 전혀 진행되지 않음

- 배치 정규화를 사용하면 학습이 빨라지며, 가중치 초깃값에 크게 의존하지 않아도 됨

### 바른 학습을 위해

- 기계학습에서는 '오버피팅'이 문제가 되는 일이 많음

- 오버피팅 : 신경망이 훈련 데이터에만 지나치게 적응되어 그 외의 데이터에는 제대로 대응하지 못하는 상태

- 기계학습은 범용 성능을 지향함
- 훈련 데이터에는 포함되지 않는, 아직 보지 못한 데이터가 주어져도 바르게 식별해내는 모델이 바람직함
- 복잡하고 표현력이 높은 모델을 만들 수는 있지만, 그만큼 오버피팅을 억제하는 기술이 중요

#### 오버피팅

- 오버피팅이 일어나는 경우

1. 매개변수가 많고 표현력이 높은 모델

2. 훈련 데이터가 적음

- 이 두 요건을 일부러 충족하여 오버피팅을 일으키기

- MINST 데이터셋의 훈련 데이터 중 300개만 사용하고, 7층 네트워크를 사용해 네트워크의 복잡성 높이기

- 각 층의 뉴런은 100개, ReLU 사용

- train세트와 test 세트의 정확도가 크게 벌어지는 것은 훈련 데이터에서 적응(fiting)해버린 결과

- 훈련 때 사용하지 않은 범용 데이터(시험 데이터)에는 제대로 대응하지 못하는 것을 이 그래프에서 확인할 수 있음

#### 가중치 감소

- 오버피팅 억제용으로 예로부터 많이 이용해온 방법 중 '가중치 감소(weight decay)라는 것이 있음

- 이는 학습 과정에서 큰 가중치에 대해서는 그에 상응하는 큰 페널티를 부과하여 오버피팅을 억제하는 방법
- 원래 오버피팅은 가중치 매개변수의 값이 커서 발생하는 경우가 많음

#### 드롭아웃

드롭아웃(Dropout)

- 뉴런을 임의로 삭제하면서 학습하는 방법

- 훈련 때 은닉층의 뉴런을 무작위로 골라 삭제함
- 삭제된 뉴런은 신호를 전달하지 않게 됨
- 훈련 때는 데이터를 흘릴 때마다 삭제할 뉴런을 무작위로 선택하고, 시험 때는 모든 뉴런에 신호를 전달
- 단, 시험 때는 각 뉴런의 출력에 훈련 때 삭제 안 한 비율을 곱하여 출력

In [16]:
class Dropout:
    def __init__(self, dropout_ratio = 0.5):
        self.dropout_ratio = dropout_ratio
        self.mask = None
    
    def forward(self, x, train_flg = True):
        if train_fig:
            self.mask = np.random.rand(*x.shape) > self.dropout_ratio
            return x * self.mask
        else:
            return x * (1.0 - self.dropout_ratio)
        
    def backward(self, dout):
        return dout * self.mask

- 훈련 시에는 순전파 때마다 self.mask에 삭제할 뉴런을 False로 표시함

- self.mask는 x와 형상이 같은 배열을 무작위로 생성하고, 그 값이 dropout_ratio보다 큰 원소만 True로 설정
- 역전파 때의 동적은 ReLU와 같음
- - 순전파 때 신호를 통과시키는 뉴런은 역전파 때도 신호를 그대로 통과시키고, 순전파 때 통과시키지 않은 뉴런은 역전파 때도 신호를 차단함

- 기계학습에서는 앙상블 학습(ensemble learning)을 애용

- 앙상블 학습은 개별적으로 학습시킨 여러 모델을 출력을 평균 내어 추론하는 방식
- 신경망의 맥락에서, 비슷한(같은) 구조의 네트워크를 5개 준비하여 따로따로 학습시키고, 시험 때는 그 5개의 출력을 평균 내어 답함
- 앙상블 학습을 수행하면 신경망의 정확도가 몇% 정도 개선된다는 것이 실험적으로 알려져 있다


- 앙상블 학습은 드롭아웃과 밀접함

- 드롭아웃이 학습 때 뉴런을 무작위로 삭제하는 행위를 매번 다른 모델을 학습시키는 것으로 해석할 수 있음
- 그리고 추론 때는 뉴런의 출력에 삭제한 비율을 곱함으로써 앙상블 학습에서 여러 모델의 평균을 내는 것과 같은 효과를 얻음
- 드롭아웃은 앙상블 학습과 같은 효과를 (대략) 하나의 네트워크로 구현